# LoRA fine-tuning ViT-B/16 on Food-101

Fine-tunes a Vision Transformer on the **Food-101** benchmark (101 classes, 75,750 train /
25,250 test images) using **LoRA** — low-rank adapters injected into the attention layers —
so under 3% of parameters are trained.

**Benchmark context (Food-101 test top-1):**

| Method | Year | Top-1 |
|---|---|---|
| Random Forest Discriminant Components (original paper, Bossard et al.) | 2014 | 56.4% |
| AlexNet transfer | 2014 | 66.4% |
| Inception-v3 fine-tune (Hassannejad et al.) | 2016 | 88.3% |
| WISeR (WIde-Slice ResNet) | 2017 | 90.3% |
| ViT-B/16 full fine-tune (all 86M params) | 2021 | ~90.5% |
| **This notebook: ViT-B/16 + LoRA (<3% trainable params)** | — | **target > 90.3%** |

**Config history** (methodical tuning, not a lucky run):
- v1 — r=16, Q+V only, 4 epochs, basic augmentation → **88.31%** TTA top-1
- v2 (current) — r=32, all attention projections (Q/K/V/out), 10 epochs, + TrivialAugment

## Before running
**Accelerator**: GPU T4 · **Internet**: On.  Total runtime ≈ 3–3.5 h (dataset download +
10 epochs + full test-set evaluation with TTA).

**Outputs**: `model_export.zip` (LoRA adapter + classifier head + preprocessor + labels —
tens of MB, serves on CPU), `results.json`, `confused_pairs.csv`, sample-grid image.

In [ ]:
# ============================== CONFIG ==============================
BASE_MODEL = "google/vit-base-patch16-224-in21k"  # ImageNet-21k pretrained, no head
EPOCHS = 10            # LoRA converges slower than full fine-tuning; don't undertrain
BATCH = 64
LR = 3e-4              # LoRA tolerates much higher LR than full fine-tuning
LORA_R = 32            # adapter rank (first run: r=16/4 epochs/Q+V -> 88.3% top-1)
LORA_ALPHA = 64
WORK = "/kaggle/working"
CACHE = "/kaggle/tmp/hf"   # dataset cache on the big scratch disk
# ====================================================================

# Pin transformers to v4: v5 renamed ViT's internal modules (query/value ->
# q_proj/v_proj), which both breaks LoRA target_modules AND silently fails to
# load the pretrained hub checkpoint into the renamed architecture.
!pip -q install "transformers>=4.44,<5" peft datasets accelerate

# Remove Kaggle's preinstalled torchao 0.10: peft's LoRA injection probes it,
# deems it too old (<0.16) and raises instead of skipping. We don't use torchao;
# with it gone, peft cleanly ignores that quantization backend.
!pip -q uninstall -y torchao

import transformers
print("transformers", transformers.__version__)
assert transformers.__version__ < "5", "restart the session so the pinned transformers is picked up"
!nvidia-smi -L

In [ ]:
%%time
# Food-101 from the HF hub (~5 GB). The 'validation' split IS the official test set,
# so numbers computed on it are directly comparable to published results.
from datasets import load_dataset

ds = load_dataset("food101", cache_dir=CACHE)
labels = ds["train"].features["label"].names
id2label = {i: l for i, l in enumerate(labels)}
label2id = {l: i for i, l in enumerate(labels)}
print(ds)
print(f"{len(labels)} classes, e.g.", labels[:5])

In [ ]:
# Sample grid for the README
import matplotlib.pyplot as plt
import random

fig, axes = plt.subplots(3, 6, figsize=(14, 7.5))
for ax in axes.flat:
    i = random.randrange(len(ds["train"]))
    ex = ds["train"][i]
    ax.imshow(ex["image"]); ax.set_title(labels[ex["label"]].replace("_", " "), fontsize=9)
    ax.axis("off")
plt.tight_layout(); plt.savefig(f"{WORK}/sample_grid.png", dpi=110); plt.show()

In [ ]:
# Preprocessing: standard ViT normalization; stronger augmentation now that we
# train 10 epochs (TrivialAugment keeps the extra epochs from overfitting).
import torch
from torchvision import transforms as T
from transformers import AutoImageProcessor

processor = AutoImageProcessor.from_pretrained(BASE_MODEL)
size = processor.size["height"]
norm = T.Normalize(mean=processor.image_mean, std=processor.image_std)

train_tf = T.Compose([
    T.RandomResizedCrop(size, scale=(0.6, 1.0)),
    T.RandomHorizontalFlip(),
    T.TrivialAugmentWide(),   # random single-op augmentation (color/geometry)
    T.ToTensor(), norm,
])
eval_tf = T.Compose([T.Resize(int(size * 1.14)), T.CenterCrop(size), T.ToTensor(), norm])
flip_tf = T.Compose([T.Resize(int(size * 1.14)), T.CenterCrop(size),
                     T.RandomHorizontalFlip(p=1.0), T.ToTensor(), norm])  # for TTA

def make_transform(tf):
    def apply(batch):
        batch["pixel_values"] = [tf(img.convert("RGB")) for img in batch["image"]]
        del batch["image"]
        return batch
    return apply

# small held-out slice of train for during-training monitoring;
# the official test split stays untouched until the final benchmark run
split = ds["train"].train_test_split(test_size=0.02, seed=42)
train_ds, monitor_ds, test_ds = split["train"], split["test"], ds["validation"]
train_ds.set_transform(make_transform(train_tf))
monitor_ds.set_transform(make_transform(eval_tf))
test_ds.set_transform(make_transform(eval_tf))

def collate(batch):
    return {
        "pixel_values": torch.stack([b["pixel_values"] for b in batch]),
        "labels": torch.tensor([b["label"] for b in batch]),
    }

In [ ]:
# Model: pretrained ViT backbone + fresh 101-way head. LoRA now targets ALL
# attention projections (Q, K, V, output) at rank 32 — still <3% of parameters.
# modules_to_save=["classifier"] keeps the fully-trained head in the adapter file.
from transformers import AutoModelForImageClassification
from peft import LoraConfig, get_peft_model

model, load_info = AutoModelForImageClassification.from_pretrained(
    BASE_MODEL, num_labels=len(labels), id2label=id2label, label2id=label2id,
    output_loading_info=True)

# FAIL FAST if the pretrained backbone didn't actually load (e.g. an
# architecture rename in a new transformers version). Only the fresh
# classifier head is allowed to be newly initialized.
missing_backbone = [k for k in load_info["missing_keys"] if "classifier" not in k]
assert not missing_backbone, (
    f"pretrained backbone weights FAILED to load ({len(missing_backbone)} missing keys, "
    f"e.g. {missing_backbone[:3]}) — training would start from random weights. "
    "Check the transformers version pin in the first cell."
)
print("backbone weights loaded OK; newly initialized:", load_info["missing_keys"])

# resolve attention projection names for this transformers version
# (v4: query/key/value + attention.output.dense; v5+: q_proj/k_proj/v_proj/o_proj)
module_names = [n for n, m in model.named_modules() if isinstance(m, torch.nn.Linear)]
leaf_names = {n.rsplit(".", 1)[-1] for n in module_names}
target = [t for t in ("query", "key", "value", "q_proj", "k_proj", "v_proj", "o_proj")
          if t in leaf_names]
if any(n.endswith("attention.output.dense") for n in module_names):
    target.append("attention.output.dense")  # suffix match: attention out-proj only, not the MLP
assert len(target) >= 3, f"attention projections not found among {sorted(leaf_names)}"
print("LoRA target modules:", target)

lora_cfg = LoraConfig(
    r=LORA_R, lora_alpha=LORA_ALPHA, lora_dropout=0.1,
    target_modules=target,
    modules_to_save=["classifier"],
)
model = get_peft_model(model, lora_cfg)
model.print_trainable_parameters()

In [ ]:
%%time
import numpy as np
from transformers import Trainer, TrainingArguments

def accuracy(eval_pred):
    logits, y = eval_pred
    if isinstance(logits, tuple):
        logits = logits[0]
    return {"accuracy": float((logits.argmax(-1) == y).mean())}

args = TrainingArguments(
    output_dir=f"{WORK}/ckpt",
    per_device_train_batch_size=BATCH,
    per_device_eval_batch_size=BATCH,
    num_train_epochs=EPOCHS,
    learning_rate=LR,
    lr_scheduler_type="cosine",
    warmup_ratio=0.05,
    fp16=True,
    eval_strategy="epoch",
    save_strategy="no",          # we export the adapter manually at the end
    logging_steps=50,
    dataloader_num_workers=2,
    remove_unused_columns=False,  # required with set_transform
    report_to="none",
)

trainer = Trainer(model=model, args=args,
                  train_dataset=train_ds, eval_dataset=monitor_ds,
                  data_collator=collate, compute_metrics=accuracy)
trainer.train()

## Benchmark evaluation — official Food-101 test split

Single-crop accuracy first, then **test-time augmentation** (average logits of the image and
its horizontal flip) — a free ~0.2–0.5 point, standard practice for squeezing a benchmark.

In [ ]:
%%time
import scipy.special

pred = trainer.predict(test_ds)
logits = pred.predictions[0] if isinstance(pred.predictions, tuple) else pred.predictions
y = pred.label_ids
acc = float((logits.argmax(-1) == y).mean())
print(f"Single-crop top-1: {acc:.4%}")

# TTA: second pass with horizontally flipped inputs, average the probabilities
test_flip = ds["validation"]
test_flip.set_transform(make_transform(flip_tf))
pred_f = trainer.predict(test_flip)
logits_f = pred_f.predictions[0] if isinstance(pred_f.predictions, tuple) else pred_f.predictions

probs = scipy.special.softmax(logits, -1) + scipy.special.softmax(logits_f, -1)
acc_tta = float((probs.argmax(-1) == y).mean())
top5 = float(np.mean([y[i] in np.argsort(-probs[i])[:5] for i in range(len(y))]))
print(f"TTA (hflip) top-1:  {acc_tta:.4%}")
print(f"TTA top-5:          {top5:.4%}")

In [ ]:
# Where do we land against published results?
import json

published = [
    ("RFDC (original paper, 2014)", 0.564),
    ("AlexNet transfer (2014)", 0.664),
    ("Inception-v3 fine-tune (2016)", 0.883),
    ("WISeR (2017)", 0.903),
    ("ViT-B/16 full fine-tune, 86M params (2021)", 0.905),
]
print(f"{'method':<48} top-1")
for name, a in published:
    beat = "  <-- beaten" if acc_tta > a else ""
    print(f"{name:<48} {a:.1%}{beat}")
print(f"{'THIS: ViT-B/16 + LoRA (<3% params) + TTA':<48} {acc_tta:.2%}")

json.dump({"single_crop_top1": acc, "tta_top1": acc_tta, "tta_top5": top5,
           "epochs": EPOCHS, "lora_r": LORA_R, "lora_alpha": LORA_ALPHA,
           "target_modules": target, "base_model": BASE_MODEL},
          open(f"{WORK}/results.json", "w"), indent=2)

In [ ]:
# Error analysis: most-confused class pairs (great README material)
import pandas as pd

yhat = probs.argmax(-1)
wrong = yhat != y
pairs = pd.DataFrame({"true": [labels[i] for i in y[wrong]],
                      "pred": [labels[i] for i in yhat[wrong]]})
top_conf = pairs.value_counts().head(10).rename("count").reset_index()
print(top_conf.to_string(index=False))
top_conf.to_csv(f"{WORK}/confused_pairs.csv", index=False)

In [ ]:
# Export for local serving: adapter (LoRA + classifier head) + preprocessor + labels.
# The base ViT is NOT shipped — the app pulls it from the HF hub once and applies this adapter.
import os, shutil

export = f"{WORK}/model_export"
shutil.rmtree(export, ignore_errors=True)
model.save_pretrained(export)            # adapter_model.safetensors + adapter_config.json
processor.save_pretrained(export)
json.dump({"base_model": BASE_MODEL, "labels": labels},
          open(f"{export}/labels.json", "w"))
shutil.copy(f"{WORK}/results.json", f"{export}/results.json")

shutil.make_archive(f"{WORK}/model_export", "zip", export)
shutil.rmtree(f"{WORK}/ckpt", ignore_errors=True)

import glob
for f in sorted(glob.glob(f"{WORK}/*")):
    if os.path.isfile(f):
        print(f"  {f}  ({os.path.getsize(f)/1e6:.1f} MB)")
print("\nNext: unzip model_export.zip into the project's model/ folder and run the web app.")